# Encoding, Scaling, Feature Selection

# Import Libraries

In [31]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math


# Create Meta Data

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



# Import Dataset

In [32]:
sample = pd.read_csv('../data/sample_submission.csv')
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [33]:
sample.set_index('carID', inplace = True)
train.set_index('carID', inplace = True)
test.set_index('carID', inplace = True)

## Define the independent variables as X and the dependent as Y

In [34]:
X = train.drop('price', axis = 1)
y = train['price']

## Feature Engineering before split

On this step of the project we create new variables that make sense and can help us predicting our target. For now the variables that we want to add are:
 - `car_age:` calculate the age of the car considering the current year and the year of his registration. This feature help us to capture depreciation, this is, older cars tend to be cheaper.
- `is_recent_model:` binary flag, 1 if the car is less than 3 years old, otherwise 0. We decided to create this feature because highlights newer cars, usually have higher resale prices.

- `mileage_category:` transforms the original continuous mileage feature into an ordered categorical variable with five usage levels according to the total number of mileage they have traveled.  tThis way it's easier to capture differences in vehicle wear and market value.

- `is_hybrid_or_eletric:` binary indicator that is 1 if the fuel type is Hybrid or Electric. Eco-friendly cars often hold higher market value.

- `is_automatic:`binary indicator that is 1 if the type of transmission is automatic or semi-automatic. Automatics typically cost more to buy and maintain.

- `fuel_eficciency_score:` Ratio of mpg / engineSize. Captures how efficiently the car converts fuel given engine capacity.

- `tax_to_engine_ratio:` Tax value divided by engine size. Indicates how expensive it is to maintain the car relative to its power.

- `tax_efficency:` mpg / tax. Measures how fuel-efficient the car is relative to its yearly tax burden.

- `paint_quality_category:` Paint quality binned into Low, Medium, High.

- `has_damage_or_low_paint:` Binary flag, 1 if the car is damaged or has low paint quality (<40%). Combines visible and declared damage into a single condition indicator.

- `is_first_owner:` binary indicator that is 1 if there were no previous owners. First-owner cars tend to have less wear, better care and higher prices.

- `ownership_ratio:` previousOwner / car_age. Captures how frequently ownership changed. Frequent changes might signal reliability issues.

To do this we'll implement some functions that help us defining this new variables.

In [40]:
def car_age_features(X, current_year=2020, threshold=3): 
    """Calculate car age from the registration year and marks whether the car is considered recent."""
    X["car_age"] = current_year - X["year"]
    X["is_recent_car"] = (X["car_age"] <= threshold).astype(int)
    return X

def mileage_category(X):
    """Creates a simple categorical feature 'mileage_category' based on the total mileage of each car."""
    bins = [0, 10_000, 50_000, 100_000, 150_000, float('inf')]
    labels = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

    X['mileage_category'] = pd.cut(X['mileage'], bins=bins, labels=labels, include_lowest=True)
    return X

def fuel_transmission_features(X):
    """Binary flags for eco-friendly and automatic cars."""
    X["is_hybrid_or_electric"] = X["fuelType"].isin(["Hybrid", "Electric"]).astype(int)
    X["is_automatic"] = X["transmission"].isin(["Automatic", "Semi-Auto"]).astype(int)
    X["fuel_efficiency_score"] = X["mpg"] / X["engineSize"]
    return X


def tax_and_efficiency_features(X):
    """Economic indicators based on tax and engine size."""
    X["tax_to_engine_ratio"] = X["tax"] / X["engineSize"]
    X["tax_efficiency"] = X["mpg"] / X["tax"]
    return X

def condition_features(X):
    """Simplify paint quality and damage info."""
    X["paintQuality_category"] = pd.cut(X["paintQuality%"],
                                         bins=[0, 40, 70, 100],
                                         labels=["Low", "Medium", "High"])
    X["has_damage_or_low_paint"] = ((X["hasDamage"] == 1) | (X["paintQuality%"] < 40)).astype(int)
    return X

def ownership_features(X):
    """Ownership-related flags."""
    X["is_first_owner"] = (X["previousOwners"] == 0).astype(int)
    X["ownership_ratio"] = np.where(X["car_age"] > 0,
                                     X["previousOwners"] / X["car_age"],
                                     0)
    return X


# Apply all feature functions
X_fe = X.copy()

X_fe = (X_fe
        .pipe(car_age_features)
        .pipe(mileage_category)
        .pipe(fuel_transmission_features)
        .pipe(tax_and_efficiency_features)
        .pipe(condition_features)
        .pipe(ownership_features)
       )

# Inspect results
print("New columns created:")
print([col for col in X_fe.columns if col not in X.columns])



print('--------------------------------------------------------------------')
print('Now we visualize the dataframe with the new features created to understand them better:')

novas_features = [col for col in X_fe.columns if col not in X.columns]
X_fe[novas_features].head(10)

New columns created:
['car_age', 'is_recent_car', 'mileage_category', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'paintQuality_category', 'has_damage_or_low_paint', 'is_first_owner', 'ownership_ratio']
--------------------------------------------------------------------
Now we visualize the dataframe with the new features created to understand them better:


,car_age,is_recent_car,mileage_category,is_hybrid_or_electric,is_automatic,fuel_efficiency_score,tax_to_engine_ratio,tax_efficiency,paintQuality_category,has_damage_or_low_paint,is_first_owner,ownership_ratio
carID,,,,,,,,,,,,
69512,4.0,0,Low,0,1,5.708634,NaN,NaN,Medium,0,0,1.000000
53000,1.0,1,Very Low,0,0,31.933333,96.666667,0.330345,Medium,0,0,1.000000
6366,1.0,1,Very Low,0,1,27.266667,96.666667,0.282069,Medium,0,0,4.000000
29021,2.0,1,Very Low,0,0,65.700000,145.000000,0.453103,Medium,0,0,-1.170153
10062,1.0,1,Very Low,0,0,28.533333,96.666667,0.295172,High,0,0,3.000000
14704,6.0,0,High,0,0,32.850000,15.000000,2.190000,High,0,1,0.000000
6924,3.0,1,Low,0,0,42.928571,14.285714,3.005000,High,0,0,1.333333
50783,3.0,1,High,0,0,43.062500,90.625000,0.475172,Medium,0,0,1.333333
67071,3.0,1,Low,0,1,31.400000,75.000000,0.418667,High,0,0,1.333333


In [41]:
X = X_fe.copy()  # Update X to include new features

## Split the dataset into train and validation

In [42]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X,y, test_size = 0.3, 
                                                  random_state = 0, 
                                                  #stratify = y- não pus esta parte como no notebook pq estava a dar erro e acho que é pq a nossa variavel y aqui não é um boolean mas sim um float
                                                  shuffle = True)

## Feature Engineering after split

On this step of the project we create new variables that make sense and can help us predicting our target. To create this new features we need to do some calculations like median and average. Because of this we are creating them after the train-validation split.

For now the variables that we want to add are:

- `brand_median_mileage:` mileage median per brand. Typical mileage level for each brand that is able to capture brand usage patterns.

- `brand_avg_engineSize`: engineSize mean per brand. Represents typical engine capacity of each brand.

- `model_median_tax:` median tax per model. Captures model-specific taxation tendencies.

- `fueltype_avg_mpg:` mpg mean peruelType. Average fuel efficiency for each fuel type.

- `brand_avg_age:` car_age mean er brand. Average car age per brand, basically compare newer against older brands.

To do this we'll implement some functions that help us defining this new variables.

In [45]:
def brand_median_mileage(df):
    df['brand_median_mileage'] = df['Brand'].map(df.groupby('Brand')['mileage'].median())
    return df

def brand_avg_engineSize(df):
    df['brand_avg_engineSize'] = df['Brand'].map(df.groupby('Brand')['engineSize'].mean())
    return df

def model_median_tax(df):
    df['model_median_tax'] = df['model'].map(df.groupby('model')['tax'].median())
    return df

def fueltype_avg_mpg(df):
    df['fueltype_avg_mpg'] = df['fuelType'].map(df.groupby('fuelType')['mpg'].mean())
    return df

def brand_avg_age(df, reference_year=2025):
    df['brand_avg_age'] = df['Brand'].map((df['car_age'] - df['year']).groupby(df['Brand']).transform('mean'))
    return df


X_train_fe = X_train.copy()
X_train_fe = (X_train_fe
           .pipe(brand_median_mileage)
           .pipe(brand_avg_engineSize)
           .pipe(model_median_tax)
           .pipe(fueltype_avg_mpg)
           .pipe(brand_avg_age))


# Inspect results
print("New columns created:")
print([col for col in X_train_fe.columns if col not in X_train.columns])



print('--------------------------------------------------------------------')
print('Now we visualize the dataframe with the new features created to understand them better:')

novas_features = [col for col in X_train_fe.columns if col not in X_train.columns]
X_train_fe[novas_features].head(10)

New columns created:
['brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age']
--------------------------------------------------------------------
Now we visualize the dataframe with the new features created to understand them better:


,brand_median_mileage,brand_avg_engineSize,model_median_tax,fueltype_avg_mpg,brand_avg_age
carID,,,,,
42520,15000.0,2.062538,145.0,58.133211,NaN
25115,17663.0,1.349668,145.0,51.329778,NaN
45113,15000.0,2.062538,145.0,58.133211,NaN
34623,17498.0,1.455900,145.0,50.963947,NaN
66409,16363.0,1.609022,145.0,47.207692,NaN
32062,17498.0,1.455900,145.0,50.963947,NaN
72102,16363.0,1.609022,145.0,50.963947,NaN
56657,18801.5,1.415679,145.0,50.963947,NaN
74356,16363.0,1.609022,145.0,58.133211,NaN
